# 03 · Random Forest vs XGBoost (Breast Cancer)

의료/생의학 tabular 데이터에서 **트리 앙상블은 여전히 강력한 베이스라인**.
딥러닝으로 넘어가기 전에, '이것을 이길 수 있는가'를 확인하는 것이 연구자의 기본 태도.

**이 노트북**:
1. Random Forest, XGBoost 비교
2. Feature importance 해석
3. 간단한 하이퍼파라미터 튜닝

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, roc_curve

np.random.seed(42)

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(f'shape: {X.shape}, 양성(benign=1) 비율: {y.mean():.2%}')
X.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 1. Random Forest

많은 의사결정 트리를 **병렬**로 학습시켜 평균을 내는 방식. 과적합에 강함.

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]
print(f'RF AUROC: {roc_auc_score(y_test, rf_proba):.4f}')

## 2. XGBoost

트리를 **순차적**으로 쌓으며 이전 오류를 보정(gradient boosting). 일반적으로 tabular 최강.

설치되지 않았다면 `pip install xgboost`.

In [ ]:
HAS_XGB = False
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        eval_metric='auc', random_state=42, n_jobs=-1,
    )
    xgb.fit(X_train, y_train)
    xgb_proba = xgb.predict_proba(X_test)[:, 1]
    print(f'XGBoost AUROC: {roc_auc_score(y_test, xgb_proba):.4f}')
    HAS_XGB = True
except ImportError:
    print('xgboost 미설치 → `pip install xgboost` 후 다시 실행하면 비교 가능')
except Exception as e:
    # macOS: libomp 필요 → `brew install libomp`
    print(f'xgboost 로드 실패: {e}')
    print('→ macOS라면 `brew install libomp` 후 다시 시도')

## 3. ROC 비교

In [ ]:
plt.figure(figsize=(6, 5))
fpr, tpr, _ = roc_curve(y_test, rf_proba)
plt.plot(fpr, tpr, label=f'RF (AUC={roc_auc_score(y_test, rf_proba):.3f})')
if HAS_XGB:
    fpr, tpr, _ = roc_curve(y_test, xgb_proba)
    plt.plot(fpr, tpr, label=f'XGB (AUC={roc_auc_score(y_test, xgb_proba):.3f})')
plt.plot([0, 1], [0, 1], 'k--', alpha=0.3)
plt.xlabel('FPR'); plt.ylabel('TPR')
plt.title('Breast Cancer — ROC')
plt.legend()
plt.show()

## 4. Feature Importance

어떤 feature가 예측에 중요했나? 의료 AI에서 특히 중요 — 설명 없으면 임상에서 안 씀.

In [ ]:
rf_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values().tail(15)
rf_imp.plot(kind='barh', figsize=(7, 5))
plt.title('RF — Top 15 feature importance')
plt.tight_layout()
plt.show()

## 5. 하이퍼파라미터 튜닝 (GridSearchCV)

'기본값으로 돌렸는데 이게 최선인가?'에 답하는 단계. 격자 위에서 다 해보고 best 고름.

In [ ]:
param_grid = {
    'n_estimators': [100, 300],
    'max_depth': [None, 5, 10],
    'min_samples_leaf': [1, 3],
}
gs = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid, cv=5, scoring='roc_auc', n_jobs=-1,
)
gs.fit(X_train, y_train)
print(f'best params: {gs.best_params_}')
print(f'best CV AUROC: {gs.best_score_:.4f}')
print(f'test AUROC   : {roc_auc_score(y_test, gs.predict_proba(X_test)[:, 1]):.4f}')

## 6. 인사이트 & 질문

**관찰**:
- Breast Cancer는 feature가 이미 잘 만들어진 데이터라 AUC 0.99+가 흔함 → 너무 잘 되는 데이터는 '튜토리얼용'임을 명심.
- 실제 임상 데이터는 AUC 0.7~0.85가 일반적.

**Layer 2(PyTorch)로 넘어가는 이유**:
- 여기까지는 feature가 이미 숫자. **이미지/텍스트/시퀀스**는 고전 ML이 못 다룸.
- UNLV Project 1 (병리 이미지), Project 3 (단백질 시퀀스) 은 딥러닝 필수.

**AI agent에 물어볼 것**:
1. "Random Forest와 Gradient Boosting의 '트리 만드는 방식' 차이를 그림으로 설명해줘"
2. "XGBoost가 tabular에서 여전히 딥러닝을 이기는 이유 최신 연구 관점으로 요약해줘"
3. "feature_importance와 SHAP의 차이는 뭐야?"

### Layer 1 끝 — 정리
이 단계에서 확실히 가져가야 할 것:
- [ ] 데이터 로드 → EDA → 전처리 → split → fit → evaluate 루프가 손에 익음
- [ ] 분류/회귀 지표의 '언제 뭘 보는지' 감각
- [ ] CV와 holdout test의 역할 구분
- [ ] 과적합을 판단하는 방법 (train vs test 성능 차이)

→ `02-pytorch-basics/`로 이동